In [ ]:
# ── Starter Cell 1: Load environment variables ──────────────────────────────
from dotenv import load_dotenv   # load_dotenv reads key=value pairs from a .env file
import os                        # os.getenv lets us read environment variables safely

load_dotenv()                    # reads .env in the current directory and sets env vars

api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key; returns None if missing

print("API key loaded:", api_key is not None)  # True = key found, False = check your .env

In [ ]:
# ── Starter Cell 2: Initialise the Anthropic client ─────────────────────────
from anthropic import Anthropic  # Anthropic is the main SDK class for all API calls

client = Anthropic(api_key=api_key)  # creates a reusable client bound to our key
                                     # passing api_key explicitly keeps the notebook
                                     # self-contained and avoids implicit env look-ups

print("Anthropic client ready")  # confirms object creation succeeded

In [ ]:
# ── Starter Cell 3: Smoke test ───────────────────────────────────────────────
response = client.messages.create(   # client.messages.create is the primary endpoint
    model="claude-haiku-4-5",        # fast, cheap model — ideal for quick tests
    # model="claude-sonnet-4-5",     # uncomment for higher-quality responses
    max_tokens=50,                   # hard cap on output length; required parameter
    messages=[{                      # messages is a list of turn dicts
        "role": "user",              # 'user' marks text coming from the human side
        "content": "Say hello in one short sentence."  # the actual prompt
    }]
)

print(response.content[0].text)     # content is a list; [0] gets the first block; .text extracts the string

# CCA Lab: Getting an API Key & Making Your First Request

**Course:** Building with the Claude API  
**Sub-module:** Getting Started  
**Lesson:** Getting an API Key

## What this lab covers
- How to obtain and securely store an Anthropic API key
- Understanding the `client.messages.create()` call anatomy
- Using system prompts to shape model behaviour
- Multi-turn conversation via explicit messages list accumulation
- The write → evaluate → improve → re-evaluate cycle with a numeric rubric
- Failure mode identification, live demos, and production framing
- Token usage tracking across the session

## CCA domains exercised
- API fundamentals · Prompt engineering · Evaluation · Production reliability

## Learning objectives
1. Securely load credentials with python-dotenv
2. Construct a valid `messages.create()` call from scratch
3. Explain when and why to use the `system` parameter
4. Build and accumulate a multi-turn messages list using the raw API
5. Score responses against a binary rubric and iterate to improvement
6. Identify and reproduce six common API failure modes
7. Read and interpret token usage across a full session

## CCA objective demonstrated: Secure credential management and environment configuration
### Section 1: Prerequisites and Environment Setup

Before calling the API we need two things:
1. **An API key** — obtained from console.anthropic.com
2. **A `.env` file** — stores the key outside of source code so it is never committed to git

**Why `.env`?** Hard-coding keys in notebooks means they appear in version history forever.  
**Why `python-dotenv`?** It reads the file and injects values into `os.environ` automatically.

### Required packages
```
pip install anthropic python-dotenv
```

### API Glossary

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `model` | string | ✅ | Which Claude model to use (e.g. `claude-haiku-4-5`) |
| `max_tokens` | int | ✅ | Hard cap on output length in tokens |
| `messages` | list[dict] | ✅ | Ordered list of `{role, content}` turn dicts |
| `system` | string | ❌ | Persistent privileged context prepended to every call |
| `temperature` | float | ❌ | Sampling randomness 0–1 (default ~1) |
| `stop_sequences` | list[str] | ❌ | Strings that halt generation when encountered |

### Response fields used in this lab

| Field | Description |
|-------|-------------|
| `response.content` | List of content blocks (usually one TextBlock) |
| `response.content[0].text` | The actual string output |
| `response.stop_reason` | Why generation stopped: `end_turn`, `max_tokens`, etc. |
| `response.usage.input_tokens` | Tokens consumed by the prompt |
| `response.usage.output_tokens` | Tokens in the model's reply |

## CCA objective demonstrated: Construct and annotate a reusable API helper function
### Section 2: The `ask()` Helper — Statement-by-Statement Walkthrough

In [ ]:
# ── Usage log: accumulates one entry per API call for later analysis ─────────
usage_log = []   # plain list; we append a dict after every call so nothing is lost

def ask(
    prompt,              # the user-facing text for this turn
    system="",           # optional persistent context; defaults to empty string
    model="claude-haiku-4-5",  # haiku is fast and cheap — good default for labs
    max_tokens=256,      # reasonable cap for most short responses
    label="unlabelled"   # human-readable tag so we can filter usage_log by call purpose
):
    """
    Send a single-turn prompt to Claude and return the text response.

    Parameters
    ----------
    prompt     : str  — The user message to send.
    system     : str  — Optional system prompt (persistent privileged context).
    model      : str  — Claude model identifier.
    max_tokens : int  — Maximum tokens Claude may generate.
    label      : str  — Tag stored in usage_log for later filtering.

    Returns
    -------
    str — The text of Claude's first content block.
    """
    # Build keyword args conditionally so we don't send an empty system string
    kwargs = {}                       # start with an empty dict of extra arguments
    if system:                        # only add 'system' if the caller supplied one
        kwargs["system"] = system     # system is a top-level kwarg, NOT inside messages
                                      # (the API treats it as a separate privileged context)

    response = client.messages.create(   # the one required SDK call
        model=model,                     # which Claude variant to use
        max_tokens=max_tokens,           # hard ceiling — generation stops here even mid-sentence
        messages=[                       # messages is always a list, even for one turn
            {"role": "user", "content": prompt}  # role must be exactly "user" or "assistant"
        ],
        **kwargs                         # unpacks system=... if it was set above
    )

    # response.content is a LIST because the API supports multiple content blocks
    # (e.g. text + tool_use). We always want the first block's text for simple calls.
    text = response.content[0].text      # [0] = first block; .text = the string payload

    stop = response.stop_reason          # 'end_turn' = natural finish; 'max_tokens' = truncated
    if stop == "max_tokens":             # warn the developer before they ship truncated output
        print(f"⚠️  [{label}] Truncated! Raise max_tokens.")

    # Append a structured record so we can analyse usage across the whole session later
    usage_log.append({
        "label": label,                           # which call this was
        "input_tokens":  response.usage.input_tokens,   # tokens consumed by the prompt
        "output_tokens": response.usage.output_tokens,  # tokens in the reply
        "stop_reason":   stop                     # why generation ended
    })

    return text   # caller gets the string; usage details are stored in usage_log


# Quick sanity check
print(ask("What is 2 + 2? Answer with just the number.", label="sanity"))

## CCA objective demonstrated: Identify and reproduce API error conditions
### Section 3: Intentional Error — Omitting `max_tokens`

`max_tokens` is the only parameter that is both **required** and easy to forget.  
The API will return a `400 Bad Request` if it is absent. Running the cell below
deliberately triggers that error so you can recognise it in production logs.

In [ ]:
# ── Intentional error demo ────────────────────────────────────────────────────
try:
    bad_response = client.messages.create(   # call without max_tokens
        model="claude-haiku-4-5",
        messages=[{"role": "user", "content": "Hello"}]
        # max_tokens is intentionally omitted to trigger a BadRequestError
    )
except Exception as e:                       # catch any SDK or HTTP error
    print(f"Error type : {type(e).__name__}")  # shows the SDK exception class
    print(f"Error detail: {e}")              # the human-readable message from the API
    print("\n✅ Error caught safely. Always include max_tokens in production.")

## CCA objective demonstrated: Apply the system parameter to shape persistent model behaviour
### Section 4: The `system` Parameter — What It Is, When to Use It, Why It Matters

The `system` parameter is a **privileged, persistent context** that the model sees
before any user message. It is processed separately from the `messages` list.

> **Principle: system context is immutable across turns — user messages cannot erase it.**  
> Even if a user says "ignore your instructions", the system prompt remains active
> because the API always prepends it ahead of the messages list.

### What goes in system vs. user turn?

| What to put in `system` | What to put in `user` turn |
|--------------------------|----------------------------|
| Persona / role ("You are a financial analyst") | The specific question or task for this turn |
| Output format rules ("Always reply in JSON") | Data or context specific to this request |
| Tone / style constraints ("Be concise, no jargon") | User-supplied variables or file contents |
| Safety guardrails ("Never reveal internal pricing") | Follow-up clarifications or corrections |

### When to use `system`
- **Persona** — "You are a senior Python engineer reviewing code."
- **Format enforcement** — "Always respond in valid JSON."
- **Tone/audience** — "Explain everything as if the reader is 12 years old."
- **Safety rails** — "Never discuss competitor products."

In [ ]:
# ── Side-by-side comparison: same question, different system prompts ──────────
question = "What is an API key?"   # identical user turn for both calls

# Call 1: no system prompt (default model voice)
no_system = ask(
    question,
    label="no-system"   # tag for usage_log
)

# Call 2: system prompt sets a strict persona and format rule
with_system = ask(
    question,
    system="You are a terse technical writer. Reply in exactly one sentence using plain English. No jargon.",
    label="with-system"   # same label strategy
)

print("=" * 60)
print("WITHOUT system prompt:")
print(no_system)
print()
print("WITH system prompt (terse technical writer):")
print(with_system)
print("=" * 60)
print("\nObservation: the system prompt constrains length AND style,")
print("demonstrating that system context shapes every output without")
print("the user needing to repeat those instructions each turn.")

## CCA objective demonstrated: Build a multi-turn conversation using explicit messages list accumulation
### Section 5: Multi-Turn Conversation — Raw API with Explicit `messages.append()`

The API is **stateless** — every call must supply the full conversation history.  
We maintain that history ourselves by appending each turn to a `messages` list.

**Architectural insight — first-line clarity:**  
The first user message sets the frame for the entire conversation. Make it
self-contained: include all context the model needs so that later turns can
build on a clear foundation rather than patching a vague opener.

**Cost implication:** Every turn re-sends the entire history as `input_tokens`.
A 10-turn conversation pays for the full context 10 times. Monitor growth.

In [ ]:
# ── Multi-turn conversation using client.messages.create() directly ───────────
# We use the raw API here (not ask()) so the messages accumulation is fully visible.

messages = []   # start with an empty history list; we own this state

# ── Turn 1 ───────────────────────────────────────────────────────────────────
# Append the first user message BEFORE the API call so the list is visible
messages.append({              # explicit append — the key mechanism the API relies on
    "role": "user",            # must be exactly "user" (not "human" — that causes a 400)
    "content": "I am learning about APIs. In one sentence, what does 'endpoint' mean?"
    # First message is self-contained: it provides full context so the model
    # can answer without needing prior turns.
})

response1 = client.messages.create(   # raw API call — no helper wrapper
    model="claude-haiku-4-5",
    max_tokens=120,
    messages=messages              # pass the full history list (only 1 turn so far)
)

reply1 = response1.content[0].text   # extract text from the first (only) content block
tokens_after_turn1 = response1.usage.input_tokens  # record before we extend the list

# Append the assistant reply so it becomes part of history for the next turn
messages.append({              # explicit append of the assistant turn
    "role": "assistant",       # must be exactly "assistant"
    "content": reply1          # the model's actual text becomes the next context
})

# Log usage for Section 8 analysis
usage_log.append({
    "label": "multiturn-turn1",
    "input_tokens":  response1.usage.input_tokens,
    "output_tokens": response1.usage.output_tokens,
    "stop_reason":   response1.stop_reason
})

print(f"Turn 1 — input_tokens: {response1.usage.input_tokens}")
print(f"Assistant: {reply1}")
print()

# ── Turn 2 ───────────────────────────────────────────────────────────────────
messages.append({              # append the follow-up user turn
    "role": "user",
    "content": "Give me one real-world example of an endpoint from a popular API."
    # This follow-up is short because Turn 1 already set the context.
})

response2 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=150,
    messages=messages          # now contains 3 entries: user, assistant, user
)

reply2 = response2.content[0].text
tokens_after_turn2 = response2.usage.input_tokens   # will be larger — history grew

messages.append({              # keep history complete for any further turns
    "role": "assistant",
    "content": reply2
})

usage_log.append({
    "label": "multiturn-turn2",
    "input_tokens":  response2.usage.input_tokens,
    "output_tokens": response2.usage.output_tokens,
    "stop_reason":   response2.stop_reason
})

print(f"Turn 2 — input_tokens: {response2.usage.input_tokens}")
print(f"Assistant: {reply2}")
print()

# ── Token growth analysis ─────────────────────────────────────────────────────
growth = tokens_after_turn2 - tokens_after_turn1
print("── Token growth across turns ───────────────────────────────")
print(f"  After Turn 1 input_tokens : {tokens_after_turn1}")
print(f"  After Turn 2 input_tokens : {tokens_after_turn2}")
print(f"  Growth                    : +{growth} tokens")
print()
print("Cost implication: each turn re-sends the full history.")
print("A 20-turn chat pays for the entire context 20 times.")
print(f"Current messages list length: {len(messages)} entries")

## CCA objective demonstrated: Score model outputs against a binary rubric and iterate to improvement
### Section 6: Improvement Cycle — Write → Evaluate → Improve → Re-Evaluate

A **rubric** converts subjective quality judgements into measurable scores.
Each criterion is **binary (0 or 1)** — either the response meets the bar or it doesn't.
Binary scoring prevents partial-credit disputes and makes automation straightforward.

The cycle forces iteration:
1. **Write** a prompt
2. **Evaluate** the response against the rubric
3. If score < threshold → **Improve** the prompt
4. **Re-evaluate** and confirm measurable gain

In [ ]:
import re   # regular expressions for pattern matching in response text

# ── Named constant: the score a response must reach to be considered production-ready
PASS_THRESHOLD = 4   # out of 4 possible points (all criteria must pass)

def score_response(text):
    """
    Score a response against four independent binary criteria.

    Each criterion returns 1 (pass) or 0 (fail) — no partial credit.
    Binary scoring makes the rubric automatable and dispute-free.

    Parameters
    ----------
    text : str — The model response to evaluate.

    Returns
    -------
    dict with keys: criteria scores, total, pass (bool)
    """
    # ── Criterion 1: Clarity ──────────────────────────────────────────────────
    # re.search scans the ENTIRE string (unlike re.match which only checks the start).
    # \b is a word boundary — it prevents 'API' inside 'RAPID' from matching.
    clarity = int(bool(
        re.search(r"\bAPI\b", text, re.IGNORECASE)  # must mention 'API'
    ))
    # int(bool(...)) converts: None→False→0, Match→True→1
    # We use int() not just bool() because we want numeric addition later.

    # ── Criterion 2: Conciseness ──────────────────────────────────────────────
    word_count = len(text.split())    # split() on whitespace gives a word list
    concise = int(word_count <= 80)   # 1 if short enough, 0 if too long

    # ── Criterion 3: Actionability — must mention numbered steps ─────────────
    # We check for at least 2 numbered steps ("1.", "2.", etc.) or step-like words.
    key_steps = ["step", "first", "then", "next", "finally",
                 r"\d+\."]   # numeric list markers like "1.", "2."

    # Generator expression: for each pattern in key_steps, test whether it appears
    # in text; yield 1 if yes, 0 if no. The generator feeds sum() WITHOUT building
    # an intermediate list in memory — more efficient for long key_steps lists.
    steps_found = sum(
        int(bool(re.search(pattern, text, re.IGNORECASE)))  # 1 or 0 per pattern
        for pattern in key_steps   # iterate: test each pattern in turn
    )                              # sum() accumulates the 1s and 0s
    actionable = int(steps_found >= 2)   # pass only if 2+ step indicators present

    # ── Criterion 4: Completeness — mentions both key and authentication ──────
    has_key  = int(bool(re.search(r"\bkey\b",            text, re.IGNORECASE)))
    has_auth = int(bool(re.search(r"\bauthenticat\w*\b", text, re.IGNORECASE)))
    complete = int(has_key == 1 and has_auth == 1)   # binary AND of two sub-checks

    scores = {
        "clarity":      clarity,
        "concise":      concise,
        "actionable":   actionable,
        "complete":     complete,
    }
    scores["total"] = sum(scores.values())              # 0–4
    scores["pass"]  = scores["total"] >= PASS_THRESHOLD # True/False verdict
    return scores


def print_scorecard(scores, label):
    """
    Print a per-criterion scorecard with ✅/❌ icons and a final PASS/FAIL verdict.

    Parameters
    ----------
    scores : dict — output of score_response()
    label  : str  — version label for display (e.g. 'V1', 'V2')
    """
    icon = {1: "✅", 0: "❌"}   # dict lookup is cleaner than if/else for binary values
    print(f"\n── Scorecard: {label} ──────────────────────────")
    for criterion in ["clarity", "concise", "actionable", "complete"]:
        s = scores[criterion]             # 0 or 1
        status = "PASS" if s else "FAIL"  # human label
        print(f"  {icon[s]} {criterion.capitalize():<12} {status} ({s}/1)")
    verdict = "PASS ✅" if scores["pass"] else "FAIL ❌"
    print(f"  {'─'*38}")
    print(f"  Total: {scores['total']}/{PASS_THRESHOLD}  →  {verdict}")


print(f"PASS_THRESHOLD = {PASS_THRESHOLD} (all 4 binary criteria must pass)")
print("score_response() and print_scorecard() defined.")

In [ ]:
# ── V1 Prompt: intentionally vague to expose rubric failures ─────────────────
PROMPT_V1 = "Tell me about API keys."   # too vague: no format, length, or content constraints

response_v1 = ask(PROMPT_V1, max_tokens=300, label="v1-prompt")

print("V1 Response:")
print(response_v1)

scores_v1 = score_response(response_v1)   # evaluate against the binary rubric
print_scorecard(scores_v1, "V1")

# ── Gating logic: only proceed to V2 if V1 fails ────────────────────────────
if scores_v1["total"] < PASS_THRESHOLD:
    print(f"\n🔁 FAIL — V1 scored {scores_v1['total']}/{PASS_THRESHOLD}. Iterating to V2...")
else:
    print(f"\n🎉 V1 already passes ({scores_v1['total']}/{PASS_THRESHOLD}). V2 still shown for comparison.")

In [ ]:
# ── V2 Prompt: specific constraints address every rubric criterion ────────────
PROMPT_V2 = (
    "Explain what an API key is and how to use one for authentication. "
    "Keep your answer under 80 words. "
    "Structure it as numbered steps (1., 2., 3.) covering: what it is, how to get it, "
    "and how to include it in a request."
)

response_v2 = ask(PROMPT_V2, max_tokens=300, label="v2-prompt")

print("V2 Response:")
print(response_v2)

scores_v2 = score_response(response_v2)   # re-evaluate the improved prompt
print_scorecard(scores_v2, "V2")

# ── Side-by-side comparison table ────────────────────────────────────────────
criteria = ["clarity", "concise", "actionable", "complete"]

print("\n── Improvement Summary Table ───────────────────────────────────")
print(f"{'Criterion':<14} {'V1':>4} {'V2':>4} {'Delta':>6} {'Status':>10}")
print("─" * 44)

for c in criteria:
    v1_score = scores_v1[c]       # score for this criterion in V1
    v2_score = scores_v2[c]       # score for this criterion in V2
    delta = v2_score - v1_score   # positive = improved, 0 = unchanged, negative = regressed

    # Nested ternary: first checks if delta is positive (improvement),
    # then checks if negative (regression), otherwise equal (unchanged).
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "=")

    status = f"{arrow} {'improved' if delta > 0 else ('regressed' if delta < 0 else 'same')}"
    print(f"{c.capitalize():<14} {v1_score:>4} {v2_score:>4} {delta:>+6} {status:>10}")

print("─" * 44)
print(f"{'TOTAL':<14} {scores_v1['total']:>4} {scores_v2['total']:>4} {scores_v2['total']-scores_v1['total']:>+6}")

## CCA objective demonstrated: Identify, reproduce, and frame production failure modes
### Section 7: Failure Mode Analysis

Every failure mode below is a **production reliability bug**, not an aesthetic preference.
In a live product, each one causes user-visible errors, billing surprises, or silent
data corruption.

| # | Failure Mode | Root Cause | Production Impact | Fix |
|---|-------------|------------|-------------------|-----|
| 1 | Vague prompt → inconsistent output | No format/length constraint | Downstream parsers break on schema mismatch | Add explicit format + length instructions |
| 2 | Missing `max_tokens` | Required parameter omitted | 400 Bad Request — request never completes | Always include `max_tokens` |
| 3 | Truncated response (`stop_reason="max_tokens"`) | `max_tokens` set too low | Silent data loss mid-sentence | Raise `max_tokens`; detect in post-processing |
| 4 | Wrong `role` value (e.g. `"human"`) | Typo or SDK confusion | 400 Bad Request — API rejects the message | Use only `"user"` or `"assistant"` |
| 5 | Consecutive same-role messages | Messages list built incorrectly | 400 Bad Request — roles must alternate | Alternate user/assistant strictly |
| 6 | Rate limit / 429 error | Too many requests per minute | Requests silently dropped; users see errors | Implement exponential back-off + retry logic |

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the **rule + judge layering**
> pattern from the CCA evaluation lab series. Deterministic rules catch obvious
> failures cheaply; the semantic judge catches subtle ones that keyword matching misses.

In [ ]:
# ── Live failure demo 1: wrong role value → 400 error ────────────────────────
print("Demo 1: invalid role value")
try:
    client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=50,
        messages=[{"role": "human", "content": "Hello"}]   # 'human' is invalid; must be 'user'
    )
except Exception as e:
    print(f"  ❌ {type(e).__name__}: {str(e)[:120]}")
    print("  Fix: use role='user' (not 'human', not 'Human')\n")

# ── Live failure demo 2: consecutive same-role messages → 400 error ───────────
print("Demo 2: consecutive same-role messages")
try:
    client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=50,
        messages=[
            {"role": "user", "content": "Hello"},
            {"role": "user", "content": "Are you there?"}   # two 'user' turns in a row
        ]
    )
except Exception as e:
    print(f"  ❌ {type(e).__name__}: {str(e)[:120]}")
    print("  Fix: alternate user/assistant strictly. Merge consecutive user messages.\n")

# ── Live failure demo 3: vague prompt → output variance ──────────────────────
# This is a PRODUCTION RELIABILITY BUG: if downstream code expects a numbered list
# but receives prose on some calls, the parser breaks intermittently.
print("Demo 3: vague prompt → output format variance (3 runs)")
vague_prompt = "Explain API keys."   # no format, no length, no structure specified

results = []
for i in range(3):                   # run 3 times to observe variance
    r = ask(vague_prompt, max_tokens=200, label=f"variance-run-{i+1}")
    word_count = len(r.split())      # count words to measure length variance
    # Detect format: does the response use numbered list markers?
    uses_list = bool(re.search(r"\b\d+\.", r))   # True if '1.' / '2.' present
    format_str = "numbered-list" if uses_list else "prose"
    results.append({"run": i+1, "words": word_count, "format": format_str})
    print(f"  Run {i+1}: {word_count} words, format={format_str}")

word_counts = [r["words"] for r in results]   # extract word counts for range calc
formats = [r["format"] for r in results]      # extract format labels

print(f"\n  Word count range: {min(word_counts)}–{max(word_counts)} words")
print(f"  Format consistency: {len(set(formats))} distinct format(s) observed: {set(formats)}")
if len(set(formats)) > 1:
    print("  ⚠️  FORMAT INCONSISTENCY DETECTED — production parser would fail intermittently.")
else:
    print("  ℹ️  Same format across runs, but word count still varies — a fixed format constraint is safer.")
print("\n  This is a production reliability bug: add explicit format instructions to the prompt.")

## CCA objective demonstrated: Track, analyse, and interpret token usage across a session
### Section 8: Token Usage Analysis

Token usage drives API cost. Tracking it across all calls lets you:
- Identify unexpectedly large prompts
- Detect truncated responses (`stop_reason == "max_tokens"`)
- Compare output token counts between prompt versions to verify specificity helps
- Forecast costs before scaling to production

In [ ]:
# ── Full session token summary ────────────────────────────────────────────────
print(f"{'#':<4} {'Label':<22} {'Input':>7} {'Output':>7} {'Total':>7} {'Stop Reason':<14}")
print("─" * 62)

total_input = 0
total_output = 0

for i, entry in enumerate(usage_log, start=1):   # enumerate gives us a 1-based call number
    inp  = entry["input_tokens"]
    out  = entry["output_tokens"]
    tot  = inp + out
    total_input  += inp
    total_output += out
    trunc = " ⚠️" if entry["stop_reason"] == "max_tokens" else ""   # flag truncation inline
    print(f"{i:<4} {entry['label']:<22} {inp:>7} {out:>7} {tot:>7} {entry['stop_reason']:<14}{trunc}")

print("─" * 62)
print(f"{'TOTAL':<4} {'':22} {total_input:>7} {total_output:>7} {total_input+total_output:>7}")

# ── Truncation scan ───────────────────────────────────────────────────────────
# Generator expression: filters usage_log entries where stop_reason is 'max_tokens'.
# We use a list comprehension (not a generator) here because we need to count and iterate.
truncated_calls = [e for e in usage_log if e["stop_reason"] == "max_tokens"]

print()
if truncated_calls:
    print(f"⚠️  TRUNCATION WARNING: {len(truncated_calls)} call(s) hit max_tokens limit:")
    for e in truncated_calls:
        print(f"   - label='{e['label']}' (output_tokens={e['output_tokens']}) — raise max_tokens")
else:
    print("✅ No truncated calls detected in this session.")

# ── V1 vs V2 output token comparison ─────────────────────────────────────────
# A more specific prompt should produce more predictable (lower-variance) output length.
v1_entry = next((e for e in usage_log if e["label"] == "v1-prompt"), None)
v2_entry = next((e for e in usage_log if e["label"] == "v2-prompt"), None)

if v1_entry and v2_entry:
    print()
    print("── V1 vs V2 Output Token Comparison ───────────────────────")
    print(f"  V1 (vague prompt)    output_tokens: {v1_entry['output_tokens']}")
    print(f"  V2 (specific prompt) output_tokens: {v2_entry['output_tokens']}")
    diff = v2_entry['output_tokens'] - v1_entry['output_tokens']
    direction = "longer" if diff > 0 else ("shorter" if diff < 0 else "same length")
    print(f"  Delta: {diff:+d} tokens ({direction})")
    print("  Insight: a constrained prompt (V2) produces more predictable output")
    print("  lengths, reducing cost variance in production batch jobs.")

## CCA objective demonstrated: Apply core concepts independently through structured drills
### Section 9: Practice Drills

Three exercises to apply the concepts from this lab. Each builds on the previous.

In [ ]:
# ── Drill 1: System prompt persona swap ───────────────────────────────────────
# Task: call ask() with a system prompt that makes Claude respond as a pirate.
# Then call it without the system prompt. Print both and compare tone.

pirate_system = "You are a pirate. Respond in pirate dialect. Keep it under 30 words."
question_d1 = "What is the internet?"

normal_answer  = ask(question_d1, label="drill1-normal")
pirate_answer  = ask(question_d1, system=pirate_system, label="drill1-pirate")

print("Drill 1 — System prompt persona swap")
print(f"Normal : {normal_answer[:200]}")
print(f"Pirate : {pirate_answer[:200]}")
print()

# ── Drill 2: Build a 3-turn conversation manually ─────────────────────────────
# Task: without using ask(), use client.messages.create() directly to hold a
# 3-turn conversation. Print each reply and the input_token count per turn.

print("Drill 2 — 3-turn raw conversation")
drill2_msgs = []
drill2_turns = [
    "What is a REST API in one sentence?",
    "Give one example endpoint from a real service.",
    "What HTTP verb would I use to create a new resource?"
]

for turn_text in drill2_turns:          # iterate over the three user messages
    drill2_msgs.append({"role": "user", "content": turn_text})  # explicit append
    r = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=100,
        messages=drill2_msgs           # always pass the full accumulated history
    )
    reply = r.content[0].text
    drill2_msgs.append({"role": "assistant", "content": reply})  # append assistant turn
    usage_log.append({"label": f"drill2-turn{len(drill2_msgs)//2}",
                      "input_tokens": r.usage.input_tokens,
                      "output_tokens": r.usage.output_tokens,
                      "stop_reason": r.stop_reason})
    print(f"  Turn {len(drill2_msgs)//2} (input_tokens={r.usage.input_tokens}): {reply[:120]}")

print()

# ── Drill 3: Extend the rubric ────────────────────────────────────────────────
# Task: add a 5th binary criterion — the response must include an example URL.
# Then score the V2 response with the extended rubric.

def score_extended(text):
    """score_response() + a 5th binary criterion: must include a URL-like example."""
    base = score_response(text)          # reuse existing 4 criteria
    has_url = int(bool(
        re.search(r"https?://\S+", text)  # looks for http:// or https:// followed by non-space
    ))
    base["has_url"]  = has_url
    base["total"]    = sum(v for k, v in base.items() if k not in ("total", "pass", "has_url")) + has_url
    base["pass"]     = base["total"] >= 5   # new threshold for 5 criteria
    return base

extended_scores = score_extended(response_v2)
print("Drill 3 — Extended rubric (5 criteria):")
print(f"  has_url : {'✅' if extended_scores['has_url'] else '❌'} ({extended_scores['has_url']}/1)")
print(f"  Total   : {extended_scores['total']}/5  Pass={extended_scores['pass']}")

## CCA objective demonstrated: Recall and apply five exam-ready mental models from this lab
### Section 10: CCA Takeaways

| # | Mental Model | Key Insight |
|---|-------------|-------------|
| 1 | **`max_tokens` is mandatory** | Omitting it raises a 400 immediately — build it into every helper by default |
| 2 | **`system` is immutable context** | Processed before `messages`; user turns cannot erase it — put stable rules there |
| 3 | **Messages list = state** | The API is stateless; you own history by appending `{role, content}` dicts explicitly |
| 4 | **Binary rubrics enable automation** | Graded scales introduce subjectivity; 0/1 criteria make improvement cycles unambiguous |
| 5 | **Token cost compounds with turns** | Each turn re-sends the full history — monitor `input_tokens` growth and design for it |

### Quick-reference: the improvement cycle

```
1. Write prompt  →  2. Call API  →  3. Score with binary rubric
       ↑                                      |
       └──────── 4. If score < PASS_THRESHOLD ─┘
                    → improve constraints
                    → re-evaluate
                    → print side-by-side table
```

### What to study next
- **CCA Evaluation Lab** — rule + judge layering pattern, Claude-as-judge semantic scoring
- **Streaming responses** — `client.messages.stream()` for real-time output
- **Tool use** — structured function calling for deterministic outputs